# Preguntas de Análisis

In [1]:
import pandas as pd

categorias = pd.read_parquet("categorias.parquet")
libros = pd.read_parquet("libros.parquet")

print("Categorías:", categorias.shape)
print("Libros:", libros.shape)

Categorías: (50, 5)
Libros: (1000, 17)


In [2]:
print("Columnas de categorias.parquet:")
print(categorias.columns.tolist())

print("\nColumnas de libros.parquet:")
print(libros.columns.tolist())

Columnas de categorias.parquet:
['categoria', 'url_categoria', 'cantidad_libros', 'fecha_extraccion', 'extraido_por']

Columnas de libros.parquet:
['upc', 'titulo', 'categoria', 'descripcion', 'tipo_producto', 'precio_sin_impuesto', 'precio_con_impuesto', 'impuesto', 'moneda', 'disponibilidad', 'cantidad_stock', 'calificacion', 'cantidad_resenas', 'url_libro', 'url_imagen', 'fecha_extraccion', 'extraido_por']


## 1. ¿Cuántas categorías de libros existen?

In [3]:
cantidad_categorias = categorias["categoria"].nunique()

print(f"Existen {cantidad_categorias} categorías de libros.")

Existen 50 categorías de libros.


## 2. ¿Cuántos libros hay en cada categoría? 

In [4]:
libros_por_categoria = (
    categorias[
        ["categoria", "cantidad_libros"]
    ]
    .sort_values(
        by="cantidad_libros",
        ascending=False
    )
    .reset_index(drop=True)
)

libros_por_categoria

,categoria,cantidad_libros
0,Default,152
1,Nonfiction,110
2,Sequential Art,75
3,Add a comment,67
4,Fiction,65
5,Young Adult,54
6,Fantasy,48
7,Romance,35
8,Mystery,32
9,Food and Drink,30


## 3. ¿Cuál es el libro más caro? Si hay varios libros con el mismo precio máximo, se deben mostrar todos. 

In [5]:
precio_maximo = libros["precio_con_impuesto"].max()

libros_mas_caros = libros.loc[
    libros["precio_con_impuesto"] == precio_maximo,
    [
        "upc",
        "titulo",
        "categoria",
        "precio_con_impuesto",
        "moneda"
    ]
].reset_index(drop=True)

print(f"Precio máximo: {precio_maximo:.2f} GBP")

libros_mas_caros

Precio máximo: 59.99 GBP


,upc,titulo,categoria,precio_con_impuesto,moneda
0,9cc207168a03470d,The Perfect Play (Play by Play #1),Romance,59.99,GBP


## 4. ¿Hay algún libro que aparezca en más de una categoría? La comparación debe realizarse utilizando el upc. 

In [6]:
categorias_por_libro = (
    libros
    .groupby("upc")
    .agg(
        titulo=("titulo", "first"),
        cantidad_categorias=("categoria", "nunique"),
        categorias=(
            "categoria",
            lambda valores: ", ".join(
                sorted(valores.unique())
            )
        )
    )
    .reset_index()
)

libros_en_varias_categorias = categorias_por_libro[
    categorias_por_libro["cantidad_categorias"] > 1
].reset_index(drop=True)

if libros_en_varias_categorias.empty:
    print(
        "No hay libros que aparezcan en más de una categoría "
        "según el UPC."
    )
else:
    print(
        f"Se encontraron {len(libros_en_varias_categorias)} "
        "libros en más de una categoría."
    )

libros_en_varias_categorias

No hay libros que aparezcan en más de una categoría según el UPC.


,upc,titulo,cantidad_categorias,categorias


## 5. ¿Cuál es el libro más barato de cada categoría? Si hay varios libros con el mismo precio mínimo, se deben mostrar todos. 

In [7]:
precio_minimo_por_categoria = (
    libros
    .groupby("categoria")["precio_con_impuesto"]
    .transform("min")
)

libros_mas_baratos = libros.loc[
    libros["precio_con_impuesto"] == precio_minimo_por_categoria,
    [
        "categoria",
        "upc",
        "titulo",
        "precio_con_impuesto",
        "moneda"
    ]
].sort_values(
    by=["categoria", "titulo"]
).reset_index(drop=True)

libros_mas_baratos

,categoria,upc,titulo,precio_con_impuesto,moneda
0,Academic,7093cf549cd2e7de,Logan Kade (Fallen Crest High #5.5),13.12,GBP
1,Add a comment,224fa77d4b248046,The Tipping Point: How Little Things Can Make ...,10.02,GBP
2,Adult Fiction,ed813a848580ba50,Fifty Shades Freed (Fifty Shades #3),15.36,GBP
3,Art,8a150fd8ff5d7686,History of Beauty,10.29,GBP
4,Autobiography,c99761a700ade23f,The Argonauts,10.93,GBP
5,Biography,d69dd8dac66f9f84,Louisa: The Extraordinary Life of Mrs. Adams,16.85,GBP
6,Business,3bebf34ee9330cbd,The Third Wave: An Entrepreneur’s Vision of th...,12.61,GBP
7,Childrens,157f693d9e600ccc,Counting Thyme,10.62,GBP
8,Christian,a322361a37a78d3d,Blue Like Jazz: Nonreligious Thoughts on Chris...,25.77,GBP
9,Christian Fiction,4807043b8218e9ce,Counted With the Stars (Out from Egypt #1),17.97,GBP


## 6. ¿Cuánto más caro o más barato es cada libro respecto al precio promedio de su categoría?

In [8]:
precio_promedio_por_categoria = (
    libros
    .groupby("categoria")["precio_con_impuesto"]
    .transform("mean")
)

diferencia_promedio = libros[
    [
        "upc",
        "titulo",
        "categoria",
        "precio_con_impuesto"
    ]
].copy()

diferencia_promedio["precio_promedio_categoria"] = (
    precio_promedio_por_categoria
)

diferencia_promedio["diferencia"] = (
    diferencia_promedio["precio_con_impuesto"]
    - diferencia_promedio["precio_promedio_categoria"]
)

diferencia_promedio[
    [
        "precio_con_impuesto",
        "precio_promedio_categoria",
        "diferencia"
    ]
] = diferencia_promedio[
    [
        "precio_con_impuesto",
        "precio_promedio_categoria",
        "diferencia"
    ]
].round(2)

diferencia_promedio

,upc,titulo,categoria,precio_con_impuesto,precio_promedio_categoria,diferencia
0,a22124811bfa8350,It's Only the Himalayas,Travel,45.17,39.79,5.38
1,ce60436f52c5ee68,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,39.79,9.64
2,f9705c362f070608,See America: A Celebration of Our National Par...,Travel,48.87,39.79,9.08
3,1809259a5a5f1d8d,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,39.79,-2.85
4,a94350ee74deaa07,Under the Tuscan Sun,Travel,37.33,39.79,-2.46
...,...,...,...,...,...,...
995,2b5054a4192e9b06,Why the Right Went Wrong: Conservatism--From G...,Politics,52.65,53.61,-0.96
996,3968e3fbf4695d7c,Equal Is Unfair: America's Misguided Fight Aga...,Politics,56.86,53.61,3.25
997,bb8245f52c7cce8f,Amid the Chaos,Cultural,36.58,36.58,0.00
998,88c21fcd38e2486e,Dark Notes,Erotica,19.19,19.19,0.00


In [9]:
def clasificar_precio(diferencia):
    if diferencia > 0:
        return "Más caro que el promedio"
    elif diferencia < 0:
        return "Más barato que el promedio"
    else:
        return "Igual al promedio"


diferencia_promedio["comparacion"] = (
    diferencia_promedio["diferencia"]
    .apply(clasificar_precio)
)

diferencia_promedio

,upc,titulo,categoria,precio_con_impuesto,precio_promedio_categoria,diferencia,comparacion
0,a22124811bfa8350,It's Only the Himalayas,Travel,45.17,39.79,5.38,Más caro que el promedio
1,ce60436f52c5ee68,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,39.79,9.64,Más caro que el promedio
2,f9705c362f070608,See America: A Celebration of Our National Par...,Travel,48.87,39.79,9.08,Más caro que el promedio
3,1809259a5a5f1d8d,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,39.79,-2.85,Más barato que el promedio
4,a94350ee74deaa07,Under the Tuscan Sun,Travel,37.33,39.79,-2.46,Más barato que el promedio
...,...,...,...,...,...,...,...
995,2b5054a4192e9b06,Why the Right Went Wrong: Conservatism--From G...,Politics,52.65,53.61,-0.96,Más barato que el promedio
996,3968e3fbf4695d7c,Equal Is Unfair: America's Misguided Fight Aga...,Politics,56.86,53.61,3.25,Más caro que el promedio
997,bb8245f52c7cce8f,Amid the Chaos,Cultural,36.58,36.58,0.00,Igual al promedio
998,88c21fcd38e2486e,Dark Notes,Erotica,19.19,19.19,0.00,Igual al promedio


## 7. Suponiendo que se venden todas las unidades disponibles, ¿qué libro produciría el mayor ingreso dentro de cada categoría? 

In [10]:
libros_ingresos = libros.copy()

libros_ingresos["ingreso_potencial"] = (
    libros_ingresos["precio_con_impuesto"]
    * libros_ingresos["cantidad_stock"]
)

libros_ingresos[
    [
        "titulo",
        "categoria",
        "precio_con_impuesto",
        "cantidad_stock",
        "ingreso_potencial"
    ]
].head()

,titulo,categoria,precio_con_impuesto,cantidad_stock,ingreso_potencial
0,It's Only the Himalayas,Travel,45.17,19,858.23
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,15,741.45
2,See America: A Celebration of Our National Par...,Travel,48.87,14,684.18
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,8,295.52
4,Under the Tuscan Sun,Travel,37.33,7,261.31


In [11]:
ingreso_maximo_por_categoria = (
    libros_ingresos
    .groupby("categoria")["ingreso_potencial"]
    .transform("max")
)

mayor_ingreso_por_categoria = libros_ingresos.loc[
    libros_ingresos["ingreso_potencial"]
    == ingreso_maximo_por_categoria,
    [
        "categoria",
        "upc",
        "titulo",
        "precio_con_impuesto",
        "cantidad_stock",
        "ingreso_potencial",
        "moneda"
    ]
].sort_values(
    by=["categoria", "titulo"]
).reset_index(drop=True)

mayor_ingreso_por_categoria["ingreso_potencial"] = (
    mayor_ingreso_por_categoria[
        "ingreso_potencial"
    ].round(2)
)

mayor_ingreso_por_categoria

,categoria,upc,titulo,precio_con_impuesto,cantidad_stock,ingreso_potencial,moneda
0,Academic,7093cf549cd2e7de,Logan Kade (Fallen Crest High #5.5),13.12,5,65.60,GBP
1,Add a comment,228f74b74f3a08ae,Judo: Seven Steps to Black Belt (an Introducto...,53.90,16,862.40,GBP
2,Adult Fiction,ed813a848580ba50,Fifty Shades Freed (Fifty Shades #3),15.36,3,46.08,GBP
3,Art,ccd9ffa25efabdea,Wall and Piece,44.18,18,795.24,GBP
4,Autobiography,825d6c44da3ca5a6,Lab Girl,40.85,11,449.35,GBP
5,Biography,5b546a46a86a9d8c,Benjamin Franklin: An American Life,48.19,7,337.33,GBP
6,Business,2597b5a345f45e1b,The Dirty Little Secrets of Getting Your Dream...,33.34,19,633.46,GBP
7,Childrens,9528d0948525bf5f,Birdsong: A Story in Pictures,54.64,19,1038.16,GBP
8,Christian,e4ac92d89b946781,(Un)Qualified: How God Uses Broken People to D...,54.00,16,864.00,GBP
9,Christian Fiction,a57b1dcbd6849222,Close to You,49.46,15,741.90,GBP


# Notas

## El archivo requirements.txt contiene la lista de librerías instaladas en el entorno virtual y sus versiones. Para eso se utilizó el comando pip freeze > requirements.txt